# TAE-IA · Module 6 · L25 — «¿Qué suena aquí?» · **the vision half, and the join**

| | |
|---|---|
| **Module** | 6 — Practical AI Applications: Images and Audio |
| **Session** | L25 — Track C, opening |
| **What this is** | The second half of Friday's skeleton. At the end of it you have **one** app. |
| **Due** | Today in class, or as homework with no penalty |
| **Graded** | No. What counts is that it runs and that you show it |

---

## What you are building today

On Friday your app learned to **listen**: CLAP tagged a clip against a vocabulary you wrote.
Today it learns to **look**, with the same vocabulary — and then it compares the two.

```
   audio clip                         photo
       |                                |
 [ coerce_audio ]                [ coerce_image ]     ← both given
       |                                |
 [ CLAP + Whisper ]                 [ CLIP ]
  your L24 half                    today's half
       |                                |
 analyse_audio(...)              analyse_image(...)
       |                                |
       +----------- same SCENES --------+
                        |
                  [ join(a, v) ]   ← do they agree?
                        |
                    [ Gradio ]
```

**CLIP is to images what CLAP is to audio**: it turns the photo into a vector, turns each of
your phrases into a vector in the same space, and answers with the closest. That is why one
vocabulary works for both — and why the join is short.

---

## The spec

Your app must, **at minimum**:

1. Contain your **L24 audio half**, working (section 2).
2. Tag an image with **CLIP** against the **same `SCENES`** your audio half uses.
3. **Refuse to guess** on an image below a floor you measured — its own floor, not CLAP's.
4. `analyse_image` returns **the same five keys** as `analyse_audio`.
5. `join` says whether the sound and the image agree — and says so honestly when either side
   was not sure enough to compare.
6. One Gradio app: `gr.Audio` **and** `gr.Image` in, a scene dropdown, one button, one report.

And it must **pass the test cell** in section 7.

## The models

| For | Model | Have you used it? |
|---|---|---|
| Tagging sound | `laion/clap-htsat-unfused` | Yes — Friday |
| Detecting speech | Whisper `small` | Yes — L19, L20, Friday |
| Tagging images | `openai/clip-vit-base-patch32` | Yes — L12's router. **New today: the `pipeline` way to call it** |

## Rules

Same as Friday. Use whatever you like, including an assistant. The test cell decides, not effort.
Anything unfinished goes home as homework with no penalty. At the end, a few of you show what
runs — finished or not.

---

## 1 — Install and setup · **given, identical to Friday**

In [1]:
# Two packages. CLAP rides inside `transformers`, which Colab already has.
!pip install -q openai-whisper librosa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 34.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [14]:
import os, sys, gc, time, random
import numpy as np
import torch

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/TAE_IA_M6'
OUTPUT_DIR = f'{DRIVE_ROOT}/L24_output'
INPUT_DIR  = f'{DRIVE_ROOT}/input_M6_L25'          # the shared folder from L11/L12
for d in (OUTPUT_DIR, INPUT_DIR):
    os.makedirs(d, exist_ok=True)

# Models live on the runtime disk, not on Drive. Wiped when the runtime recycles
# (~2 min to re-download), and in exchange there is no OSError 95 from symlinks.
MODEL_CACHE = '/content/models'
os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ['HF_HOME']        = MODEL_CACHE
os.environ['TORCH_HOME']     = MODEL_CACHE
os.environ['XDG_CACHE_HOME'] = MODEL_CACHE

if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > T4 GPU')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def vram(tag=''):
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM {used:5.2f} / {total:.1f} GB   {tag}')

print(torch.cuda.get_device_name(0)); vram('empty')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tesla T4
  VRAM  2.21 / 15.6 GB   empty


## 2 — Your audio half, from Friday

Three cells below are **yours to paste**, from your own L24 notebook — exactly as they ran
there. The contract cell between them is given again, unchanged, so paste around it.

| Paste into | From your L24 notebook |
|---|---|
| **2a** | section 2 — loading CLAP (`tagger`) and Whisper (`asr`) |
| **2b** | section 4 — `SCENES` and `CONF_FLOOR` (and anything else you defined there) |
| **2c** | section 5 — `tag_audio`, `transcribe`, `analyse_audio`, `run_audio` |

> **Did not finish Friday?** Then this is where today starts: finish those three cells first,
> here. The vision half needs `SCENES` to exist, and the join needs `analyse_audio` to work.

The check cell after them tells you whether your half survived the trip.

In [15]:
# 2a — Audio models from the completed L24 notebook.
from transformers import pipeline
import whisper

tagger = pipeline(
    'zero-shot-audio-classification',
    model='laion/clap-htsat-unfused',
    device=0
)
vram('after CLAP')

asr = whisper.load_model('small', download_root=MODEL_CACHE)
vram('both audio models loaded')

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

  VRAM  2.21 / 15.6 GB   after CLAP
  VRAM  2.21 / 15.6 GB   both audio models loaded


### The audio contract · **given, unchanged from Friday**

In [16]:
import librosa

# The rate each stage wants. Two different numbers in one pipeline.
CLAP_SR     = 48000     # measured: at any other rate CLAP fails intermittently
ASR_SR      = 16000     # Whisper
MAX_SECONDS = 60        # a 30-minute upload is the #1 cause of a stalled demo
MIN_SECONDS = 0.5


def list_inputs():
    """What is in the shared folder right now."""
    names = sorted(f for f in os.listdir(INPUT_DIR) if not f.startswith('.'))
    print(f'{INPUT_DIR}  ({len(names)} files)')
    for n in names:
        print(f'  {os.path.getsize(os.path.join(INPUT_DIR, n))/1e6:6.2f} MB  {n}')
    return names


def upload_inputs():
    """Pick files from your machine; they land in Drive and stay there."""
    from google.colab import files
    for fname, data in files.upload().items():
        with open(os.path.join(INPUT_DIR, fname), 'wb') as f:
            f.write(data)
        print(f'  saved {fname}')


def load_audio(name_or_path, sr=None):
    """A name in the inputs folder, or any path -> (wav float32 mono, sr)."""
    path = name_or_path
    if not os.path.isabs(path):
        path = os.path.join(INPUT_DIR, name_or_path)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{name_or_path} is not in {INPUT_DIR}. '
            'Run upload_inputs() and pick it, or copy it into that folder on Drive.')
    wav, got_sr = librosa.load(path, sr=sr, mono=True)
    return wav.astype('float32'), got_sr


def coerce_audio(wav, sr, target_sr):
    """Anything a user can hand us -> float32 mono at target_sr, or ValueError.

    This function is the contract. Call it ALWAYS, before a model sees anything.
    """
    if wav is None:
        raise ValueError('No audio received. Please upload a clip.')
    wav = np.asarray(wav)
    if wav.dtype.kind in 'iu':                       # gradio hands back int16
        wav = wav.astype('float32') / np.iinfo(wav.dtype).max
    wav = wav.astype('float32')
    if wav.ndim > 1:                                 # to mono, either layout
        wav = wav.mean(axis=0) if wav.shape[0] < wav.shape[1] else wav.mean(axis=1)
    if wav.size < sr * MIN_SECONDS:
        raise ValueError(f'Clip too short ({wav.size/sr:.2f} s). Use at least {MIN_SECONDS} s.')
    if wav.size > sr * MAX_SECONDS:
        wav = wav[:int(sr * MAX_SECONDS)]            # truncate, and the caller says so
    if np.abs(wav).max() < 1e-4:
        raise ValueError('That clip is silent. Whisper would invent words for it.')
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
    return wav.astype('float32'), target_sr


print('contract loaded. CLAP_SR =', CLAP_SR, '| ASR_SR =', ASR_SR)

contract loaded. CLAP_SR = 48000 | ASR_SR = 16000


In [17]:
# 2b — The same vocabulary and measured floor used in L24.
SCENES = {
    'casa': [
        'a dog barking',
        'a baby crying',
        'a clock ticking',
        'a vacuum cleaner',
        'a person speaking',
    ],
    'exterior': [
        'rain falling',
        'birds chirping',
        'a chainsaw',
        'a car engine',
        'a person speaking',
    ],
}

ES = {
    'a dog barking': 'un perro ladrando',
    'a baby crying': 'un bebé llorando',
    'a clock ticking': 'un reloj haciendo tic-tac',
    'a vacuum cleaner': 'una aspiradora',
    'a person speaking': 'una persona hablando',
    'rain falling': 'lluvia',
    'birds chirping': 'pájaros cantando',
    'a chainsaw': 'una motosierra',
    'a car engine': 'un motor de coche',
}

CONF_FLOOR = 0.60
SPEECH_MIN_WORDS = 3

In [18]:
# 2c — Audio functions from the completed L24 notebook.
def tag_audio(wav, sr, labels):
    assert sr == CLAP_SR, f'{sr} != {CLAP_SR} - coerce first'
    output = tagger(wav, candidate_labels=list(labels))
    return [(item['label'], float(item['score'])) for item in output]


def transcribe(wav, sr, language=None):
    assert sr == ASR_SR, f'{sr} != {ASR_SR} - coerce first'
    result = asr.transcribe(wav, language=language, fp16=True)
    return result['text'].strip()


def analyse_audio(wav, sr, scene):
    if scene not in SCENES:
        raise ValueError(f'Escena desconocida: {scene!r}.')

    w48, _ = coerce_audio(wav, sr, CLAP_SR)
    tags = tag_audio(w48, CLAP_SR, SCENES[scene])

    w16, _ = coerce_audio(wav, sr, ASR_SR)
    transcript = transcribe(w16, ASR_SR)
    speech = len(transcript.split()) >= SPEECH_MIN_WORDS

    top_label, top_score = tags[0]
    if top_score >= CONF_FLOOR:
        summary = f'Suena a {ES.get(top_label, top_label)}.'
    else:
        summary = 'No pude identificar este sonido con seguridad.'
    if speech:
        summary += f' Alguien dice: «{transcript}»'

    return {
        'modality': 'audio',
        'scene': scene,
        'tags': tags,
        'text': transcript if speech else None,
        'summary': summary,
    }


def run_audio(x, sr=None, scene='casa'):
    try:
        if isinstance(x, str):
            wav, sr = load_audio(x)
        else:
            wav = x
            if sr is None:
                raise ValueError('Falta la tasa de muestreo del audio.')
        started = time.time()
        result = analyse_audio(wav, sr, scene)
        elapsed = time.time() - started
    except (ValueError, FileNotFoundError) as error:
        return str(error), str(error)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        message = 'La GPU se quedó sin memoria. Prueba con un clip más corto.'
        return message, message
    except Exception as error:
        print(f'[run_audio] {type(error).__name__}: {error}')
        message = 'Algo salió mal. Prueba con otro clip.'
        return message, message

    top3 = '\n'.join(
        f'  {label:22s} {score:.3f}' for label, score in result['tags'][:3]
    )
    report = (
        f"{result['summary']}\n\n"
        f"escena : {result['scene']}\n"
        f"top-3  :\n{top3}\n"
        f"voz    : {result['text'] or '—'}\n"
        f"tiempo : {elapsed:.2f} s"
    )
    return result['summary'], report

In [19]:
# ---------- check: did the audio half survive the trip? (given) ----------
assert tagger is not None and asr is not None, 'cell 2a: load both models'
assert SCENES, 'cell 2b: SCENES is empty'
assert isinstance(CONF_FLOOR, (int, float)), 'cell 2b: CONF_FLOOR must be a number you measured'

probe = [f for f in sorted(os.listdir(INPUT_DIR)) if f.lower().endswith(('.wav', '.mp3', '.flac', '.ogg'))]
wav, sr = load_audio(probe[0]) if probe else \
          ((0.2 * np.random.default_rng(SEED).standard_normal(CLAP_SR * 5)).astype('float32'), CLAP_SR)
a = analyse_audio(wav, sr, next(iter(SCENES)))
KEYS = {'modality', 'scene', 'tags', 'text', 'summary'}
assert set(a) >= KEYS, f'analyse_audio is missing keys: {KEYS - set(a)}'
assert a['modality'] == 'audio'
print('audio half OK on', probe[0] if probe else 'white noise', '->', a['summary'])
vram('audio half loaded')

audio half OK on birds.wav -> No pude identificar este sonido con seguridad.
  VRAM  2.21 / 15.6 GB   audio half loaded


## 3 — The image contract · **given**

Friday's contract was the sample rate. Today's is the one from L11: **anything a user can hand
the app becomes an RGB PIL image of a sane size**, or a `ValueError` that says what to fix.

Two things Gradio can hand you that are not what CLIP wants: a PNG with an alpha channel
(RGBA), and a black-and-white photo (mode `L`). Both have to become RGB before the model sees
them. The same function also stops a 6000-pixel phone photo from eating the GPU.

In [20]:
from PIL import Image

MAX_SIDE, MIN_SIDE = 1024, 32
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.webp')


def load_image(name_or_path):
    """A name in the inputs folder, or any path -> PIL image (not yet coerced)."""
    path = name_or_path
    if not os.path.isabs(path):
        path = os.path.join(INPUT_DIR, name_or_path)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{name_or_path} is not in {INPUT_DIR}. '
            'Run upload_inputs() and pick it, or copy it into that folder on Drive.')
    return Image.open(path)


def coerce_image(x):
    """Anything a user can hand us -> RGB PIL image, longest side <= MAX_SIDE, or ValueError.

    This is the L11 contract, unchanged. Call it ALWAYS, before a model sees anything.
    """
    if x is None:
        raise ValueError('No image received. Please upload one.')
    if isinstance(x, str):
        x = load_image(x)
    if isinstance(x, np.ndarray):
        x = Image.fromarray(x.astype(np.uint8))
    if not isinstance(x, Image.Image):
        raise ValueError('Unsupported input. Please upload an image file.')
    x = x.convert('RGB')                              # RGBA, L, P, CMYK -> RGB
    w, h = x.size
    if min(w, h) < MIN_SIDE:
        raise ValueError(f'Image too small ({w}x{h}). Use at least {MIN_SIDE} px.')
    if max(w, h) > MAX_SIDE:
        s = MAX_SIDE / max(w, h)
        x = x.resize((int(w * s), int(h * s)), Image.LANCZOS)
    return x


print('image contract loaded. MAX_SIDE =', MAX_SIDE)

image contract loaded. MAX_SIDE = 1024


## 4 — CLIP, and its floor · **you write them**

In L12 you called CLIP by hand: a processor, a model, `get_image_features`, a cosine
similarity. The `pipeline` does all of that in one line, and — this is the point — **its call
looks exactly like your CLAP call**:

```python
tagger(wav, candidate_labels=labels)          # Friday, audio
vision_tagger(img, candidate_labels=labels)   # today, images
```

The task is `zero-shot-image-classification`; the repo is `openai/clip-vit-base-patch32`.
Same output shape as CLAP: a list of dicts with `label` and `score`, highest first.

Three measured things worth knowing, on the images in the shared folder:

- **Your audio phrases work for images as they are.** The pipeline quietly wraps each one as
  `"This is a photo of {phrase}"` and hands your own label back. `'a dog barking'` found the
  dog photo at 0.98. You do not need a second vocabulary — that is the whole premise.
- **CLIP's scores are not on CLAP's scale.** A floor you measured for sound is not a floor for
  images. Measure this one separately, the same way: what does a hit score, what does a miss.
- **The confident mistake is back.** A photo of a bird, in a scene with no bird label, came back
  *"a person speaking"* at **0.96**. Exactly Friday's lesson: no floor catches that. The fix is
  still the vocabulary.

In [21]:
from transformers import pipeline

# CLIP uses the same candidate-label interface as CLAP.
vision_tagger = pipeline(
    'zero-shot-image-classification',
    model='openai/clip-vit-base-patch32',
    device=0
)

vram('after CLIP')        # expect ~0.6 GB more

# Initial image threshold for the shared image pack. Recheck it with your own photos.
IMAGE_CONF_FLOOR = 0.60

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  VRAM  2.21 / 15.6 GB   after CLIP


## 5 — The vision half · **you write it**

Three functions, deliberately the mirror image of Friday's. The signatures are given so the
join and the test cell can rely on them.

`analyse_image` returns **the same five keys** as `analyse_audio`. `modality` is `'image'`, and
`text` is `None` — there is nothing to transcribe in a photo.

In [22]:
def tag_image(img, labels):
    """PIL RGB image + a list of phrases -> [(label, score), ...] sorted, highest first.

    Assumes img has already been through coerce_image.
    """
    output = vision_tagger(img, candidate_labels=list(labels))
    return [(item['label'], float(item['score'])) for item in output]


def analyse_image(image, scene):
    """Anything a user can hand us + a scene name -> the same five-key dict as analyse_audio.

    {'modality': 'image', 'scene': scene, 'tags': [...], 'text': None, 'summary': '...'}
    Apply IMAGE_CONF_FLOOR, and write `summary` in Spanish.
    """
    if scene not in SCENES:
        raise ValueError(f'Escena desconocida: {scene!r}.')

    img = coerce_image(image)
    tags = tag_image(img, SCENES[scene])
    top_label, top_score = tags[0]

    if top_score >= IMAGE_CONF_FLOOR:
        summary = f'La imagen parece mostrar {ES.get(top_label, top_label)}.'
    else:
        summary = 'No pude identificar la imagen con seguridad.'

    return {
        'modality': 'image',
        'scene': scene,
        'tags': tags,
        'text': None,
        'summary': summary,
    }


def run_image(x, scene):
    """The image half's UI entry point. Anything in, and it NEVER raises -> (summary, report)."""
    try:
        started = time.time()
        result = analyse_image(x, scene)
        elapsed = time.time() - started
    except (ValueError, FileNotFoundError) as error:
        return str(error), str(error)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        message = 'La GPU se quedó sin memoria. Prueba con una imagen más pequeña.'
        return message, message
    except Exception as error:
        print(f'[run_image] {type(error).__name__}: {error}')
        message = 'Algo salió mal. Prueba con otra imagen.'
        return message, message

    top3 = '\n'.join(
        f'  {label:22s} {score:.3f}' for label, score in result['tags'][:3]
    )
    report = (
        f"{result['summary']}\n\n"
        f"escena : {result['scene']}\n"
        f"top-3  :\n{top3}\n"
        f"tiempo : {elapsed:.2f} s"
    )
    return result['summary'], report

## 6 — The join · **you write it**

This is the multimodal part, and it is short on purpose: both halves return the same shape
against the same vocabulary, so comparing them is comparing two lists.

The one decision that matters is **what to say when a side was not sure**. If the audio refused
to guess, "they disagree" would be a lie — the honest answer is "I cannot compare". That is why
`agree` has three values, and the test cell checks all three.

In [23]:
def join(a, v):
    """analyse_audio's dict + analyse_image's dict -> the verdict.

    Returns {'agree': True | False | None, 'audio': label | None, 'image': label | None,
             'summary': one sentence in Spanish}

      agree  True   both sides cleared their floor and named the same label
             False  both cleared their floor and named different labels
             None   either side was below its floor (CONF_FLOOR / IMAGE_CONF_FLOOR),
                    so there is nothing honest to compare
      audio / image: the top label of that side, or None if it was below its floor
    """
    if a.get('scene') != v.get('scene'):
        raise ValueError('Audio e imagen deben usar la misma escena.')

    audio_label, audio_score = a['tags'][0]
    image_label, image_score = v['tags'][0]
    audio_ok = audio_score >= CONF_FLOOR
    image_ok = image_score >= IMAGE_CONF_FLOOR

    named_audio = audio_label if audio_ok else None
    named_image = image_label if image_ok else None

    if not audio_ok or not image_ok:
        agree = None
        summary = 'No puedo comparar: al menos una entrada no fue identificada con seguridad.'
    elif audio_label == image_label:
        agree = True
        summary = f'El audio y la imagen coinciden: {ES.get(audio_label, audio_label)}.'
    else:
        agree = False
        summary = (
            f'El audio y la imagen no coinciden: el audio parece {ES.get(audio_label, audio_label)} '
            f'y la imagen parece {ES.get(image_label, image_label)}.'
        )

    return {
        'agree': agree,
        'audio': named_audio,
        'image': named_image,
        'summary': summary,
    }


def run_both(audio, image, scene):
    """The one function the app's button calls. NEVER raises -> (summary, report).

    `audio` is Gradio's (sr, wav) tuple or None; `image` is a PIL image or None. If only one is
    given, report that half on its own and say that both are needed to compare.
    """
    try:
        if scene not in SCENES:
            raise ValueError(f'Escena desconocida: {scene!r}.')

        audio_result = None
        image_result = None

        if audio is not None:
            sr, wav = audio
            audio_result = analyse_audio(wav, sr, scene)
        if image is not None:
            image_result = analyse_image(image, scene)

        if audio_result is None and image_result is None:
            message = 'Sube un audio, una imagen o ambos.'
            return message, message
        if audio_result is None:
            message = image_result['summary'] + ' Falta el audio para comparar.'
            return message, message
        if image_result is None:
            message = audio_result['summary'] + ' Falta la imagen para comparar.'
            return message, message

        verdict = join(audio_result, image_result)
        audio_top = audio_result['tags'][0]
        image_top = image_result['tags'][0]
        report = (
            f"{verdict['summary']}\n\n"
            f"audio : {audio_top[0]} ({audio_top[1]:.3f})\n"
            f"imagen: {image_top[0]} ({image_top[1]:.3f})\n"
            f"escena: {scene}"
        )
        return verdict['summary'], report

    except (ValueError, FileNotFoundError) as error:
        return str(error), str(error)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        message = 'La GPU se quedó sin memoria. Usa entradas más pequeñas.'
        return message, message
    except Exception as error:
        print(f'[run_both] {type(error).__name__}: {error}')
        message = 'Algo salió mal. Prueba con otras entradas.'
        return message, message

## 7 — The test cell · **given. It has to pass.**

Four checks: hostile images never raise, the two halves return the same shape, the join tells
the truth in all three cases, and your images fire on your vocabulary.

In [24]:
scene = next(iter(SCENES))
fails = 0

# ---------- A. hostile images: none of these may raise ----------
print('--- hostile images ---')
cases = [
    ('None',          None),
    ('10x10 px',      Image.new('RGB', (10, 10), 'white')),
    ('6000x4000 px',  Image.new('RGB', (6000, 4000), 'steelblue')),
    ('grayscale L',   Image.new('L', (300, 300), 128)),
    ('RGBA',          Image.new('RGBA', (300, 300), (0, 128, 0, 100))),
    ('numpy uint8',   np.zeros((240, 320, 3), dtype=np.uint8)),
    ('missing file',  'no_such_photo.jpg'),
]
for tag, bad in cases:
    try:
        _s, msg = run_image(bad, scene)
        print(f'  {tag:14s} -> {(msg or _s or "").splitlines()[0][:60]}')
    except Exception as e:
        print(f'  {tag:14s} -> RAISED {type(e).__name__}: {e}')
        fails += 1

# ---------- B. the two halves speak the same shape ----------
print('\n--- shapes ---')
KEYS = {'modality', 'scene', 'tags', 'text', 'summary'}
v = analyse_image(Image.new('RGB', (256, 256), 'steelblue'), scene)
assert set(v) >= KEYS, f'analyse_image is missing keys: {KEYS - set(v)}'
assert v['modality'] == 'image' and v['text'] is None
assert {t for t, _ in v['tags']} == set(SCENES[scene]), 'image tags must use the SAME vocabulary'
print('  analyse_image returns the five keys, on the same vocabulary')

# ---------- C. the join tells the truth ----------
print('\n--- join ---')
L1, L2 = SCENES[scene][0], SCENES[scene][1]
def fake(modality, label, score):
    rest = [l for l in SCENES[scene] if l != label]
    tags = [(label, score)] + [(l, (1 - score) / len(rest)) for l in rest]
    return {'modality': modality, 'scene': scene, 'text': None, 'summary': '',
            'tags': sorted(tags, key=lambda t: -t[1])}          # the contract: highest first
for name, a_, v_, want in [
        ('same label',      fake('audio', L1, 0.99), fake('image', L1, 0.99), True),
        ('different label', fake('audio', L1, 0.99), fake('image', L2, 0.99), False),
        ('audio unsure',    fake('audio', L1, 0.01), fake('image', L1, 0.99), None),
        ('image unsure',    fake('audio', L1, 0.99), fake('image', L1, 0.01), None)]:
    j = join(a_, v_)
    ok = j.get('agree') is want and {'agree', 'audio', 'image', 'summary'} <= set(j)
    fails += not ok
    print(f'  {name:16s} -> agree={j.get("agree")!s:5s} {"ok" if ok else "*** expected " + str(want)}  {j.get("summary", "")[:50]}')

# ---------- D. your images against your vocabulary ----------
print('\n--- your images ---')
imgs = [f for f in sorted(os.listdir(INPUT_DIR)) if f.lower().endswith(IMAGE_EXTS)]
if not imgs:
    print('  (no images in the folder - run upload_inputs())')
for name in imgs[:6]:
    s, _ = run_image(name, scene)
    print(f'  {name}: {s}')

vram('peak')
print(f"\npeak {torch.cuda.max_memory_allocated()/1e9:.2f} GB of "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print('\nfailures:', fails, '(must be 0)')

--- hostile images ---
  None           -> No image received. Please upload one.
  10x10 px       -> Image too small (10x10). Use at least 32 px.
  6000x4000 px   -> No pude identificar la imagen con seguridad.
  grayscale L    -> No pude identificar la imagen con seguridad.
  RGBA           -> No pude identificar la imagen con seguridad.
  numpy uint8    -> No pude identificar la imagen con seguridad.
  missing file   -> no_such_photo.jpg is not in /content/drive/MyDrive/TAE_IA_M6

--- shapes ---
  analyse_image returns the five keys, on the same vocabulary

--- join ---
  same label       -> agree=True  ok  El audio y la imagen coinciden: un perro ladrando.
  different label  -> agree=False ok  El audio y la imagen no coinciden: el audio parece
  audio unsure     -> agree=None  ok  No puedo comparar: al menos una entrada no fue ide
  image unsure     -> agree=None  ok  No puedo comparar: al menos una entrada no fue ide

--- your images ---
  birds.jpg: La imagen parece mostrar una pe

## 8 — The app · **you write it**

One interface for the whole thing. Everything you need you have used before: `Blocks`, `Row`,
`Column`, `Audio`, `Image`, `Dropdown`, `Button`, `Textbox`, `.click()`, `.queue()`, `.launch()`.

The two contracts meet here for the first time, so write them down before you wire anything:

```python
gr.Audio(type='numpy')   # the callback receives (sr, wav) — rate FIRST
gr.Image(type='pil')     # the callback receives a PIL image, or None
```

Minimum requirements:

- `gr.Audio(type='numpy')` **and** `gr.Image(type='pil')` as inputs
- a `gr.Dropdown` for the scene, fed from `SCENES`
- **one** button that runs `run_both`
- a `gr.Textbox` with the report
- `.queue()` before `.launch(share=True)`

Tabs (`gr.Tab`) are allowed — you saw them in the demo — but one screen is simpler, and the
point of this app is seeing both answers side by side.

In [25]:
import gradio as gr
print('gradio', gr.__version__)

try:
    demo.close()        # re-running this cell: stop the previous app first, or it keeps its port
except NameError:
    pass

def on_click(audio, image, scene):
    return run_both(audio, image, scene)


with gr.Blocks(title='Audio + Vision Matcher') as demo:
    gr.Markdown(
        '# ¿Coinciden el sonido y la imagen?\n'
        'Sube un audio y una foto. CLAP y CLIP usan el mismo vocabulario.'
    )

    scene_input = gr.Dropdown(
        choices=list(SCENES),
        value=next(iter(SCENES)),
        label='Escena'
    )

    with gr.Row():
        audio_input = gr.Audio(type='numpy', label='Audio')
        image_input = gr.Image(type='pil', label='Imagen')

    compare_button = gr.Button('Comparar', variant='primary')
    summary_output = gr.Textbox(label='Resultado', lines=3)
    report_output = gr.Textbox(label='Reporte', lines=8)

    compare_button.click(
        fn=on_click,
        inputs=[audio_input, image_input, scene_input],
        outputs=[summary_output, report_output]
    )

demo.queue().launch(share=True)

gradio 6.26.0
Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1692661db30a7b548d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [26]:
# Stop the app when you are done with it.
# demo.close()

---

## If you get stuck

**Test each half alone before the join.** `run_audio('dog.wav', scene=...)` and
`run_image('dog.jpg', scene)` should each make sense on their own. If one of them does not,
`join` cannot fix it.

**If the image half names everything the same thing**, print `tag_image(...)` for two very
different photos. If the scores barely move, you are probably passing the same image every
time — or an image that never went through `coerce_image`.

**If the join says "no coinciden" on a dog barking and a dog photo**, print both `tags[0]`.
Labels are compared as strings: `'a dog barking'` and `'a dog barking '` are different labels.

## The showcase

Show what runs, finished or not:

1. Pick a scene, and show one pair that **agrees** — a sound and a photo of the same thing.
2. Show one pair that **disagrees**, and say whether the app was right to say so.
3. Show one case where the app **refused** — and one where it was **confidently wrong**. Say
   which label was missing from your vocabulary.

Number 3 is the one that shows you understood the app. Anyone can demo a success.

## Before you leave

- [X] The test cell reports **0** failures
- [X] `IMAGE_CONF_FLOOR` is a number you **measured**, separately from `CONF_FLOOR`
- [X] The app takes a clip **and** a photo and says whether they agree
- [X] **Restart the runtime and run everything again**

---
*TAE-IA 2025 · Cocyten-Nayarit · Module 6 · L25*